[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/08-prompt-and-context-engineering/notebooks/02-structured-output-and-reasoning.ipynb)

# 08.02 — Structured output and reasoning prompts

**What you will learn**
- Why programs need **structured output** (data in a fixed shape, usually **JSON**) instead of prose.
- How to define a **schema** with **Pydantic** (a Python library that checks data has the right fields and types).
- A **validate-and-retry** loop: parse the reply, and if it is broken, send the error back and ask again.
- **Label scoring**: a trick for local models that *guarantees* one of the allowed answers.
- **Chain-of-thought** prompts with `<thinking>` / `<answer>` tags, and **self-consistency** (majority vote).
- What the real API calls for structured output look like.

Runs offline in about 5 minutes.

## Set-up: one function to talk to a model

`call_llm(system, user, examples=None)` sends a **system prompt** (standing rules), optional **few-shot examples** (worked input/output pairs, sent as earlier chat turns) and the **user message** (the request), and returns the reply text.

- If an `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` **environment variable** (a setting stored outside your code) exists, it calls the real API.
- Otherwise it uses **SmolLM2-135M-Instruct**, a tiny free model that runs on a CPU. It is weak, which is useful: weak models make the effect of good prompting easy to see.

The first run downloads about 270 MB.

In [1]:
import os, warnings
warnings.filterwarnings("ignore")
from functools import lru_cache

LOCAL_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-5")  # model names change: check the provider's models page
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

def _messages(user, examples):
    msgs = []
    for ex_in, ex_out in (examples or []):           # few-shot pairs become fake earlier turns
        msgs += [{"role": "user", "content": ex_in}, {"role": "assistant", "content": ex_out}]
    return msgs + [{"role": "user", "content": user}]

@lru_cache(maxsize=1)
def _local():
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(LOCAL_MODEL)
    model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL).eval()
    return tok, model

def call_llm(system, user, examples=None, max_tokens=60, temperature=0.0, seed=0):
    if os.getenv("ANTHROPIC_API_KEY"):                # real Claude call (only if you set a key)
        import anthropic
        r = anthropic.Anthropic().messages.create(model=ANTHROPIC_MODEL, max_tokens=max_tokens,
                temperature=temperature, system=system, messages=_messages(user, examples))
        return r.content[0].text
    if os.getenv("OPENAI_API_KEY"):                   # real OpenAI call (only if you set a key)
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model=OPENAI_MODEL, max_tokens=max_tokens, temperature=temperature,
                messages=[{"role": "system", "content": system}] + _messages(user, examples))
        return r.choices[0].message.content
    import torch                                       # offline stand-in: tiny local model
    tok, model = _local()
    msgs = [{"role": "system", "content": system}] + _messages(user, examples)
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    torch.manual_seed(seed)                            # fixed seed = repeatable demos; change it to see variety
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=temperature > 0,
                             temperature=temperature if temperature > 0 else None, top_p=None, top_k=None,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

BACKEND = ("Anthropic API" if os.getenv("ANTHROPIC_API_KEY") else
           "OpenAI API" if os.getenv("OPENAI_API_KEY") else
           "LOCAL offline stand-in (SmolLM2-135M) - results are illustrative")
print("Backend:", BACKEND)
print(call_llm("You are a helpful assistant. Answer in one short sentence.", "What is a prompt?", max_tokens=30))

Backend: LOCAL offline stand-in (SmolLM2-135M) - results are illustrative


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 10003.43it/s]

A prompt is a brief, concise statement or question that encourages the user to think critically or make a decision. It can be a question, a problem


## 1. A schema with Pydantic

A **schema** is like a paper form with labelled boxes. We describe the boxes as a Python class. `Literal[...]` means "only one of these exact values" (an **enum**). `Field(ge=1, le=5)` means "a number from 1 to 5".

In [2]:
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError

class TicketInfo(BaseModel):
    category: Literal["billing", "delivery", "product", "account"]
    urgency: int = Field(ge=1, le=5, description="1 = can wait, 5 = urgent")
    order_id: Optional[str] = None          # None allowed: the model must not invent an order number

good = '{"category": "delivery", "urgency": 4, "order_id": "A1234"}'
bad = '{"category": "shipping", "urgency": 9}'
print("good ->", TicketInfo.model_validate_json(good))
try:
    TicketInfo.model_validate_json(bad)
except ValidationError as e:
    print("bad  ->", e.error_count(), "errors:")
    for err in e.errors():
        print("   ", err["loc"], err["msg"])

good -> category='delivery' urgency=4 order_id='A1234'
bad  -> 2 errors:
    ('category',) Input should be 'billing', 'delivery', 'product' or 'account'
    ('urgency',) Input should be less than or equal to 5


Pydantic tells us *exactly* what is wrong. That error text is useful: we can hand it back to the model.

Pydantic can also produce a **JSON Schema** (a standard way of describing JSON shapes) from the class. Real APIs accept this schema for their structured-output features.

In [3]:
import json
print(json.dumps(TicketInfo.model_json_schema(), indent=1)[:700])

{
 "properties": {
  "category": {
   "enum": [
    "billing",
    "delivery",
    "product",
    "account"
   ],
   "title": "Category",
   "type": "string"
  },
  "urgency": {
   "description": "1 = can wait, 5 = urgent",
   "maximum": 5,
   "minimum": 1,
   "title": "Urgency",
   "type": "integer"
  },
  "order_id": {
   "anyOf": [
    {
     "type": "string"
    },
    {
     "type": "null"
    }
   ],
   "default": null,
   "title": "Order Id"
  }
 },
 "required": [
  "category",
  "urgency"
 ],
 "title": "TicketInfo",
 "type": "object"
}


## 2. Validate-and-retry

The loop:
1. Ask for JSON (describe the fields and give an example).
2. Pull out the `{...}` part of the reply (models sometimes add chatter around it).
3. Validate with Pydantic.
4. If it fails, send the error message back and ask for a corrected version. Stop after a few tries.

First we show the loop with a **scripted fake model** so you can see each step clearly: its first reply is broken, its second is fixed.

In [4]:
import re

JSON_SYSTEM = """Extract ticket information as JSON with keys:
category (one of billing, delivery, product, account), urgency (integer 1-5), order_id (string or null).
Reply with JSON only. Example: {"category": "billing", "urgency": 2, "order_id": null}"""

def extract_json(text):
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)          # first { ... last }
    return m.group(0) if m else text

def structured_call(llm, ticket, max_tries=3, verbose=True):
    user = f"Ticket: {ticket}"
    for attempt in range(1, max_tries + 1):
        reply = llm(JSON_SYSTEM, user)
        try:
            result = TicketInfo.model_validate_json(extract_json(reply))
            if verbose: print(f"attempt {attempt}: OK ->", result)
            return result
        except ValidationError as e:
            if verbose: print(f"attempt {attempt}: invalid ({e.error_count()} errors) reply was: {reply[:80]!r}")
            user = (f"Ticket: {ticket}\nYour previous reply was invalid: {e.errors()[0]['msg']}. "
                    "Reply again with corrected JSON only.")
    return None                                              # give up: let the caller handle it

scripted = iter(['Sure! {"category": "shipping", "urgency": "high"}',
                 '{"category": "delivery", "urgency": 4, "order_id": "B77"}'])
fake_llm = lambda system, user: next(scripted)              # returns the next scripted reply
structured_call(fake_llm, "Order B77 is a week late and I need it for a trip tomorrow!")

attempt 1: invalid (2 errors) reply was: 'Sure! {"category": "shipping", "urgency": "high"}'
attempt 2: OK -> category='delivery' urgency=4 order_id='B77'


TicketInfo(category='delivery', urgency=4, order_id='B77')

Now the same loop with the real (tiny local or API) model. A 135M model often fails JSON tasks — watch what happens. If it gives up and returns `None`, that is the loop doing its job: **never pass unvalidated data downstream**.

In [5]:
real_llm = lambda system, user: call_llm(system, user, max_tokens=60)
for t in ["Order B77 is a week late and I need it tomorrow!", "I was charged twice, please fix it"]:
    print("TICKET:", t)
    print("RESULT:", structured_call(real_llm, t, max_tries=2), "\n")

TICKET: Order B77 is a week late and I need it tomorrow!


attempt 1: OK -> category='billing' urgency=2 order_id='B77'
RESULT: category='billing' urgency=2 order_id='B77' 

TICKET: I was charged twice, please fix it


attempt 1: invalid (1 errors) reply was: "I'm sorry for the misunderstanding, but as an AI, I don't have the ability to ac"


attempt 2: OK -> category='billing' urgency=2 order_id=None
RESULT: category='billing' urgency=2 order_id=None 



## 3. Label scoring — guaranteed valid labels on a local model

When you run a model yourself you can do something most chat APIs do not let you do: look at the **probability** the model gives to each possible answer. Instead of letting it write freely, we ask "how likely is the reply `billing`? `delivery`? ..." and pick the most likely. The output is *always* one of the allowed labels. This is the idea behind **constrained decoding** (only allowing certain tokens).

How: for each label, add it after the prompt, run the model once, and add up the **log-probabilities** (logarithms of probabilities — adding them is the same as multiplying probabilities) of the label's tokens.

In [6]:
import torch

def label_scores(system, user, labels, examples=None):
    tok, model = _local()
    msgs = [{"role": "system", "content": system}] + _messages(user, examples)
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    prompt_ids = tok(prompt, return_tensors="pt").input_ids
    scores = {}
    for label in labels:
        label_ids = tok(label, add_special_tokens=False, return_tensors="pt").input_ids
        ids = torch.cat([prompt_ids, label_ids], dim=1)
        with torch.no_grad():
            logits = model(ids).logits                          # a score for every possible next token
        logprobs = torch.log_softmax(logits[0, :-1], dim=-1)  # turn scores into log-probabilities
        positions = range(prompt_ids.shape[1] - 1, ids.shape[1] - 1)
        scores[label] = sum(logprobs[p, ids[0, p + 1]].item() for p in positions)
    return scores

SYSTEM = """You sort customer support tickets for an online shop.
Allowed categories: billing, delivery, product, account."""
LABEL_LIST = ["billing", "delivery", "product", "account"]
FEW_SHOT = [("Ticket: My card was charged the wrong amount", "billing"),
            ("Ticket: The box arrived two weeks late", "delivery"),
            ("Ticket: The stove does not light", "product"),
            ("Ticket: I forgot my username", "account")]
CASES = [("I was charged twice for the same order", "billing"), ("My parcel has not arrived after 10 days", "delivery"),
         ("The zip on the jacket broke on day one", "product"), ("I cannot log in, password reset email never comes", "account"),
         ("Please refund the extra fee on my card", "billing"), ("Courier left the box at the wrong house", "delivery"),
         ("How do I change the email on my profile?", "account"), ("The tent poles are shorter than described", "product")]

# Calibration: how much does the model like each label when the ticket says nothing at all?
baseline = label_scores(SYSTEM, "Ticket: N/A", LABEL_LIST, FEW_SHOT)
raw_ok = cal_ok = 0
for t, expected in CASES:
    s = label_scores(SYSTEM, f"Ticket: {t}", LABEL_LIST, FEW_SHOT)
    raw = max(s, key=s.get)
    cal = max(LABEL_LIST, key=lambda l: s[l] - baseline[l])      # subtract the label's built-in bias
    raw_ok += raw == expected; cal_ok += cal == expected
    print(f"{t[:42]:<44} raw={raw:<9} calibrated={cal:<9} expected={expected}")
print(f"\nraw accuracy {raw_ok}/8, calibrated accuracy {cal_ok}/8 - and the format is valid 8/8 by construction")

I was charged twice for the same order       raw=product   calibrated=delivery  expected=billing


My parcel has not arrived after 10 days      raw=delivery  calibrated=delivery  expected=delivery


The zip on the jacket broke on day one       raw=product   calibrated=product   expected=product


I cannot log in, password reset email neve   raw=account   calibrated=account   expected=account


Please refund the extra fee on my card       raw=billing   calibrated=billing   expected=billing


Courier left the box at the wrong house      raw=account   calibrated=delivery  expected=delivery


How do I change the email on my profile?     raw=account   calibrated=billing   expected=account


The tent poles are shorter than described    raw=product   calibrated=product   expected=product

raw accuracy 6/8, calibrated accuracy 6/8 - and the format is valid 8/8 by construction


Every answer is now a valid label (format is 100% by construction). We also used **calibration**: small models often prefer one label no matter what (here usually "billing"), so we subtract each label's score on a content-free ticket ("N/A"). In our runs, label scoring with few-shot examples got 6 of 8 right, versus 3 of 8 for free text generation in notebook 01 — same tiny model, better technique. Calibration helped most without examples (2/8 → 4/8 in our zero-shot tests); with the few-shot examples it made no difference here. Hosted APIs offer the same *format guarantee* through their structured-output / JSON-schema features instead.

## 4. Reasoning prompts: think first, answer second

**Chain-of-thought (CoT)** means asking the model to write its reasoning before the answer. Putting the reasoning and the answer in separate tags lets your code read only the answer.

The tiny model is too weak to reason well, so we show the *prompt* and the *parsing* with a reply written the way a capable model would answer. With an API key set, the next cell calls the real model instead.

In [7]:
COT_SYSTEM = """You check whether a customer can return an item.
Policy: returns allowed within 30 days of delivery; items must be unused; sale items are final.
Think step by step inside <thinking> tags. Then write only YES or NO inside <answer> tags."""
question = "Delivered 20 days ago, unused, bought in the summer sale. Can I return it?"

if os.getenv("ANTHROPIC_API_KEY") or os.getenv("OPENAI_API_KEY"):
    reply = call_llm(COT_SYSTEM, question, max_tokens=300)
else:   # illustrative reply in the shape a capable model produces (labelled, not generated)
    reply = ("<thinking>Delivered 20 days ago, inside the 30-day window. Unused, so condition is fine. "
             "But it was a sale item, and sale items are final.</thinking>\n<answer>NO</answer>")
    print("[offline: showing an illustrative reply, not a model output]")

def parse_answer(text):
    m = re.search(r"<answer>\s*(.*?)\s*</answer>", text, flags=re.DOTALL)
    return m.group(1) if m else None

print(reply)
print("\nYour program only reads ->", parse_answer(reply))

[offline: showing an illustrative reply, not a model output]
<thinking>Delivered 20 days ago, inside the 30-day window. Unused, so condition is fine. But it was a sale item, and sale items are final.</thinking>
<answer>NO</answer>

Your program only reads -> NO


**Why it helps:** each written step becomes context for the next, like showing your working in a maths exam. **Caveats:** reasoning costs tokens (time and money), and nicely written reasoning can still reach a wrong answer. Many providers now have "reasoning" or "extended thinking" models where you set a thinking budget instead of writing "think step by step" — check their docs.

## 5. Self-consistency: ask several times, take the majority

Another reasoning trick: sample several answers at a temperature above 0 and take the most common one. Random mistakes tend to cancel out. Here we do it on a classification with the few-shot prompt, discarding samples that are not valid labels. We use a low temperature (0.3): with a tiny model, at 0.8 almost every sample was junk ("revenue", "chargeback") in our tests — itself a useful lesson about sampling.

In [8]:
from collections import Counter
FEW_SHOT = [("Ticket: My card was charged the wrong amount", "billing"),
            ("Ticket: The box arrived two weeks late", "delivery"),
            ("Ticket: The stove does not light", "product"),
            ("Ticket: I forgot my username", "account")]
STRICT = SYSTEM + "\nReply with exactly one category word in lowercase and nothing else."

def sample_once(seed, text):
    torch.manual_seed(seed)
    if os.getenv("ANTHROPIC_API_KEY") or os.getenv("OPENAI_API_KEY"):
        return call_llm(STRICT, text, examples=FEW_SHOT, max_tokens=5, temperature=0.3)
    tok, model = _local()
    msgs = [{"role": "system", "content": STRICT}] + _messages(text, FEW_SHOT)
    inp = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=5, do_sample=True, temperature=0.3, top_p=None, top_k=None,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

text = "Ticket: Please refund the extra fee on my card"
replies = [sample_once(s, text) for s in range(5)]                 # 5 samples with different random seeds
votes = [r.split()[0].strip(".,").lower() if r.strip() else "" for r in replies]
valid = [v for v in votes if v in LABEL_LIST]                       # throw away samples that broke the format
print("5 samples:", votes)
print("valid votes:", valid)
print("majority vote ->", Counter(valid).most_common(1)[0][0] if valid else "no valid votes - fall back to label scoring")

5 samples: ['billing', 'payment', 'billing', 'billing', 'billing']
valid votes: ['billing', 'billing', 'billing', 'billing']
majority vote -> billing


## 6. The real API versions (read, do not run without a key)

Both big providers can enforce a schema for you. The code below only runs if a key is set. **Feature names and parameters change** — check the official docs linked in `resources.md` before using it.

In [9]:
if os.getenv("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    # The SDK's parse helper accepts a Pydantic class and returns a validated object.
    resp = client.chat.completions.parse(
        model=OPENAI_MODEL,
        messages=[{"role": "system", "content": "Extract ticket info."},
                  {"role": "user", "content": "Order B77 is late, I need it tomorrow!"}],
        response_format=TicketInfo)
    print(resp.choices[0].message.parsed)
elif os.getenv("ANTHROPIC_API_KEY"):
    import anthropic
    client = anthropic.Anthropic()
    # Tool use with an input schema: the model must "call" the tool with arguments matching the schema.
    resp = client.messages.create(
        model=ANTHROPIC_MODEL, max_tokens=300,
        tools=[{"name": "record_ticket", "description": "Save ticket info",
                "input_schema": TicketInfo.model_json_schema()}],
        tool_choice={"type": "tool", "name": "record_ticket"},
        messages=[{"role": "user", "content": "Order B77 is late, I need it tomorrow!"}])
    print(TicketInfo.model_validate(resp.content[0].input))
else:
    print("No API key set - skipped. (This is expected in the classroom.)")

No API key set - skipped. (This is expected in the classroom.)


## Recap
- Programs need **structured output**; describe it with a **Pydantic schema**.
- Always **validate** in code; on failure, feed the error back and **retry** a few times, then give up safely.
- Locally, **label scoring** guarantees a valid label; APIs offer schema-enforced output.
- **Chain-of-thought** with `<thinking>`/`<answer>` tags helps multi-step tasks; parse only the answer.
- **Self-consistency** = sample several times, take the majority.

**Try it:** add a `sentiment: Literal["positive", "neutral", "negative"]` field to `TicketInfo`, update `JSON_SYSTEM`, and rerun section 2 with the scripted fake model (write your own two scripted replies).